# Coordinate Routine Validation

This notebook tests the SRT controller's coordinate routines against astropy to verify accuracy.

In [ ]:
# Install astropy if needed
# !pip install astropy

In [ ]:
import math
from datetime import datetime, timezone

# Astropy imports
from astropy.coordinates import SkyCoord, EarthLocation, AltAz, get_sun, get_body
from astropy.time import Time
from astropy import units as u
import numpy as np

## Copy of SRT Coordinate Routines

These are Python ports of the routines in `esp32_controller_arduino/src/coordinates.cpp`.

In [ ]:
def julian_date(year, month, day, hour, minute, second):
    """Calculate Julian Date from calendar date/time (UTC)"""
    if month <= 2:
        year -= 1
        month += 12

    A = int(year / 100)
    B = 2 - A + int(A / 4)

    jd = int(365.25 * (year + 4716)) + int(30.6001 * (month + 1)) + day + B - 1524.5
    jd += (hour + minute / 60 + second / 3600) / 24

    return jd


def gmst(jd):
    """Calculate Greenwich Mean Sidereal Time in hours from Julian Date.

    Uses the IAU 1982 formula for GMST.
    """
    # Calculate JD at 0h UT (midnight) for this day
    jd0 = math.floor(jd - 0.5) + 0.5

    # Hours since 0h UT
    H = (jd - jd0) * 24.0

    # Days since J2000.0 at 0h UT
    D0 = jd0 - 2451545.0
    T = D0 / 36525.0

    # GMST at 0h UT in hours (IAU 1982 formula)
    gmst0 = 6.697374558 + 0.06570982441908 * D0 + 1.00273790935 * H + 0.000026 * T**2

    # Normalize to 0-24 hours
    gmst_hours = gmst0 % 24

    return gmst_hours


def local_sidereal_time(jd, longitude):
    """Calculate Local Sidereal Time in hours"""
    lst = gmst(jd) + longitude / 15.0
    return lst % 24


def precess_j2000_to_date(ra_hours, dec_deg, jd):
    """
    Precess J2000.0 coordinates to the given Julian Date.

    Uses the IAU 1976 precession model (Lieske et al.).
    Accurate to ~1 arcsec for dates within a few centuries of J2000.

    Args:
        ra_hours: Right Ascension in hours (J2000)
        dec_deg: Declination in degrees (J2000)
        jd: Target Julian Date

    Returns:
        (ra_hours, dec_deg) at the target epoch
    """
    # Julian centuries from J2000.0
    T = (jd - 2451545.0) / 36525.0

    # Precession angles in arcseconds (IAU 1976)
    zeta_A = (2306.2181 + 1.39656 * T - 0.000139 * T**2) * T + \
             (0.30188 - 0.000344 * T) * T**2 + 0.017998 * T**3
    z_A = (2306.2181 + 1.39656 * T - 0.000139 * T**2) * T + \
          (1.09468 + 0.000066 * T) * T**2 + 0.018203 * T**3
    theta_A = (2004.3109 - 0.85330 * T - 0.000217 * T**2) * T - \
              (0.42665 + 0.000217 * T) * T**2 - 0.041833 * T**3

    # Convert to radians
    zeta = math.radians(zeta_A / 3600)
    z = math.radians(z_A / 3600)
    theta = math.radians(theta_A / 3600)

    # Original coordinates in radians
    ra0 = math.radians(ra_hours * 15)
    dec0 = math.radians(dec_deg)

    # Apply precession rotation
    A = math.cos(dec0) * math.sin(ra0 + zeta)
    B = math.cos(theta) * math.cos(dec0) * math.cos(ra0 + zeta) - math.sin(theta) * math.sin(dec0)
    C = math.sin(theta) * math.cos(dec0) * math.cos(ra0 + zeta) + math.cos(theta) * math.sin(dec0)

    # New coordinates
    ra_rad = math.atan2(A, B) + z
    dec_rad = math.asin(C)

    # Convert back to hours and degrees
    ra_new = (math.degrees(ra_rad) / 15) % 24
    dec_new = math.degrees(dec_rad)

    return ra_new, dec_new

In [ ]:
def get_sun_position(dt):
    """
    Calculate the Sun's RA/Dec (J2000) for a given datetime.
    Based on Meeus "Astronomical Algorithms" - accurate to ~0.01 degree.
    """
    jd = julian_date(dt.year, dt.month, dt.day, dt.hour, dt.minute, dt.second)

    # Julian centuries since J2000.0
    T = (jd - 2451545.0) / 36525.0

    # Geometric mean longitude of the Sun (degrees)
    L0 = (280.46646 + 36000.76983 * T + 0.0003032 * T**2) % 360

    # Mean anomaly of the Sun (degrees)
    M = (357.52911 + 35999.05029 * T - 0.0001537 * T**2) % 360
    M_rad = math.radians(M)

    # Equation of center (degrees)
    C = ((1.914602 - 0.004817 * T - 0.000014 * T**2) * math.sin(M_rad) +
         (0.019993 - 0.000101 * T) * math.sin(2 * M_rad) +
         0.000289 * math.sin(3 * M_rad))

    # Sun's true longitude (degrees)
    sun_lon = L0 + C

    # Apparent longitude (corrected for nutation and aberration)
    omega = 125.04 - 1934.136 * T
    sun_lon_apparent = sun_lon - 0.00569 - 0.00478 * math.sin(math.radians(omega))
    sun_lon_rad = math.radians(sun_lon_apparent)

    # Mean obliquity of the ecliptic
    eps0 = 23.439291 - 0.0130042 * T - 0.00000016 * T**2 + 0.000000504 * T**3

    # Corrected obliquity
    eps = eps0 + 0.00256 * math.cos(math.radians(omega))
    eps_rad = math.radians(eps)

    # Convert ecliptic to equatorial coordinates
    ra_rad = math.atan2(
        math.cos(eps_rad) * math.sin(sun_lon_rad),
        math.cos(sun_lon_rad)
    )
    dec_rad = math.asin(math.sin(eps_rad) * math.sin(sun_lon_rad))

    ra_apparent = (math.degrees(ra_rad) / 15) % 24
    dec_apparent = math.degrees(dec_rad)

    # Precess from apparent (equinox of date) back to J2000
    ra_j2000, dec_j2000 = precess_date_to_j2000(ra_apparent, dec_apparent, jd)

    return ra_j2000, dec_j2000


def precess_date_to_j2000(ra_hours, dec_deg, jd):
    """
    Precess coordinates from the given Julian Date back to J2000.0.
    """
    # Julian centuries from J2000.0
    T = (jd - 2451545.0) / 36525.0

    # Precession angles in arcseconds (IAU 1976)
    zeta_A = (2306.2181 + 1.39656 * T - 0.000139 * T**2) * T + \
             (0.30188 - 0.000344 * T) * T**2 + 0.017998 * T**3
    z_A = (2306.2181 + 1.39656 * T - 0.000139 * T**2) * T + \
          (1.09468 + 0.000066 * T) * T**2 + 0.018203 * T**3
    theta_A = (2004.3109 - 0.85330 * T - 0.000217 * T**2) * T - \
              (0.42665 + 0.000217 * T) * T**2 - 0.041833 * T**3

    # Convert to radians
    zeta = math.radians(zeta_A / 3600)
    z = math.radians(z_A / 3600)
    theta = math.radians(theta_A / 3600)

    # Current coordinates in radians
    ra = math.radians(ra_hours * 15)
    dec = math.radians(dec_deg)

    # Apply inverse precession rotation
    A = math.cos(dec) * math.sin(ra - z)
    B = math.cos(theta) * math.cos(dec) * math.cos(ra - z) + math.sin(theta) * math.sin(dec)
    C = -math.sin(theta) * math.cos(dec) * math.cos(ra - z) + math.cos(theta) * math.sin(dec)

    # J2000 coordinates
    ra0_rad = math.atan2(A, B) - zeta
    dec0_rad = math.asin(C)

    # Convert back to hours and degrees
    ra0 = (math.degrees(ra0_rad) / 15) % 24
    dec0 = math.degrees(dec0_rad)

    return ra0, dec0

In [ ]:
def get_moon_position(dt):
    """
    Calculate the Moon's RA/Dec (J2000) for a given datetime.
    Based on Meeus "Astronomical Algorithms" Ch. 47 - accurate to ~0.3 degree.
    """
    jd = julian_date(dt.year, dt.month, dt.day, dt.hour, dt.minute, dt.second)

    # Julian centuries since J2000.0
    T = (jd - 2451545.0) / 36525.0

    # Moon's mean longitude (degrees)
    Lp = (218.3164477 + 481267.88123421 * T - 0.0015786 * T**2 +
          T**3 / 538841 - T**4 / 65194000) % 360

    # Moon's mean elongation (degrees)
    D = (297.8501921 + 445267.1114034 * T - 0.0018819 * T**2 +
         T**3 / 545868 - T**4 / 113065000) % 360

    # Sun's mean anomaly (degrees)
    M = (357.5291092 + 35999.0502909 * T - 0.0001536 * T**2 +
         T**3 / 24490000) % 360

    # Moon's mean anomaly (degrees)
    Mp = (134.9633964 + 477198.8675055 * T + 0.0087414 * T**2 +
          T**3 / 69699 - T**4 / 14712000) % 360

    # Moon's argument of latitude (degrees)
    F = (93.2720950 + 483202.0175233 * T - 0.0036539 * T**2 -
         T**3 / 3526000 + T**4 / 863310000) % 360

    # Additional arguments
    A1 = (119.75 + 131.849 * T) % 360
    A2 = (53.09 + 479264.290 * T) % 360
    A3 = (313.45 + 481266.484 * T) % 360

    # Eccentricity correction
    E = 1 - 0.002516 * T - 0.0000074 * T**2

    # Convert to radians
    D_rad = math.radians(D)
    M_rad = math.radians(M)
    Mp_rad = math.radians(Mp)
    F_rad = math.radians(F)
    A1_rad = math.radians(A1)
    A2_rad = math.radians(A2)
    A3_rad = math.radians(A3)

    # Sum of longitude terms (most significant terms from Table 47.A)
    sum_l = (
        6288774 * math.sin(Mp_rad) +
        1274027 * math.sin(2*D_rad - Mp_rad) +
        658314 * math.sin(2*D_rad) +
        213618 * math.sin(2*Mp_rad) +
        -185116 * E * math.sin(M_rad) +
        -114332 * math.sin(2*F_rad) +
        58793 * math.sin(2*D_rad - 2*Mp_rad) +
        57066 * E * math.sin(2*D_rad - M_rad - Mp_rad) +
        53322 * math.sin(2*D_rad + Mp_rad) +
        45758 * E * math.sin(2*D_rad - M_rad) +
        -40923 * E * math.sin(M_rad - Mp_rad) +
        -34720 * math.sin(D_rad) +
        -30383 * E * math.sin(M_rad + Mp_rad) +
        15327 * math.sin(2*D_rad - 2*F_rad) +
        -12528 * math.sin(Mp_rad + 2*F_rad) +
        10980 * math.sin(Mp_rad - 2*F_rad) +
        10675 * math.sin(4*D_rad - Mp_rad) +
        10034 * math.sin(3*Mp_rad) +
        8548 * math.sin(4*D_rad - 2*Mp_rad) +
        -7888 * E * math.sin(2*D_rad + M_rad - Mp_rad) +
        -6766 * E * math.sin(2*D_rad + M_rad) +
        -5163 * math.sin(D_rad - Mp_rad) +
        4987 * E * math.sin(D_rad + M_rad) +
        4036 * E * math.sin(2*D_rad - M_rad + Mp_rad)
    )

    # Additional longitude corrections
    Lp_rad = math.radians(Lp)
    sum_l += (
        3958 * math.sin(A1_rad) +
        1962 * math.sin(Lp_rad - F_rad) +
        318 * math.sin(A2_rad)
    )

    # Sum of latitude terms (most significant from Table 47.B)
    sum_b = (
        5128122 * math.sin(F_rad) +
        280602 * math.sin(Mp_rad + F_rad) +
        277693 * math.sin(Mp_rad - F_rad) +
        173237 * math.sin(2*D_rad - F_rad) +
        55413 * math.sin(2*D_rad - Mp_rad + F_rad) +
        46271 * math.sin(2*D_rad - Mp_rad - F_rad) +
        32573 * math.sin(2*D_rad + F_rad) +
        17198 * math.sin(2*Mp_rad + F_rad) +
        9266 * math.sin(2*D_rad + Mp_rad - F_rad) +
        8822 * math.sin(2*Mp_rad - F_rad) +
        -8216 * E * math.sin(2*D_rad - M_rad - F_rad) +
        4324 * math.sin(2*D_rad - 2*Mp_rad - F_rad) +
        4200 * math.sin(2*D_rad + Mp_rad + F_rad) +
        -3359 * E * math.sin(2*D_rad + M_rad - F_rad) +
        2463 * E * math.sin(2*D_rad - M_rad - Mp_rad + F_rad) +
        2211 * E * math.sin(2*D_rad - M_rad + F_rad) +
        2065 * E * math.sin(2*D_rad - M_rad - Mp_rad - F_rad) +
        -1870 * E * math.sin(M_rad - Mp_rad - F_rad)
    )

    # Additional latitude corrections
    sum_b += (
        -2235 * math.sin(Lp_rad) +
        382 * math.sin(A3_rad) +
        175 * math.sin(A1_rad - F_rad) +
        175 * math.sin(A1_rad + F_rad) +
        127 * math.sin(Lp_rad - Mp_rad) +
        -115 * math.sin(Lp_rad + Mp_rad)
    )

    # Ecliptic longitude and latitude (degrees)
    ecl_lon = Lp + sum_l / 1000000
    ecl_lat = sum_b / 1000000

    ecl_lon_rad = math.radians(ecl_lon)
    ecl_lat_rad = math.radians(ecl_lat)

    # Mean obliquity of the ecliptic
    eps = 23.439291 - 0.0130042 * T
    eps_rad = math.radians(eps)

    # Convert ecliptic to equatorial
    x_ecl = math.cos(ecl_lat_rad) * math.cos(ecl_lon_rad)
    y_ecl = math.cos(ecl_lat_rad) * math.sin(ecl_lon_rad)
    z_ecl = math.sin(ecl_lat_rad)

    x_eq = x_ecl
    y_eq = y_ecl * math.cos(eps_rad) - z_ecl * math.sin(eps_rad)
    z_eq = y_ecl * math.sin(eps_rad) + z_ecl * math.cos(eps_rad)

    ra_rad = math.atan2(y_eq, x_eq)
    dec_rad = math.asin(z_eq)

    ra_apparent = (math.degrees(ra_rad) / 15) % 24
    dec_apparent = math.degrees(dec_rad)

    # Precess from apparent (equinox of date) back to J2000
    ra_j2000, dec_j2000 = precess_date_to_j2000(ra_apparent, dec_apparent, jd)

    return ra_j2000, dec_j2000

In [ ]:
def ra_dec_to_alt_az(ra_hours, dec_deg, lat_deg, lon_deg, dt):
    """
    Convert RA/Dec (J2000) to Alt/Az for a given datetime and location.
    """
    jd = julian_date(dt.year, dt.month, dt.day, dt.hour, dt.minute, dt.second)

    # Precess from J2000 to current date
    ra_now, dec_now = precess_j2000_to_date(ra_hours, dec_deg, jd)

    # Calculate hour angle
    lst = local_sidereal_time(jd, lon_deg)
    ha_hours = lst - ra_now
    ha_rad = math.radians(ha_hours * 15)

    # Convert to radians
    dec_rad = math.radians(dec_now)
    lat_rad = math.radians(lat_deg)

    # Calculate altitude
    sin_alt = (math.sin(dec_rad) * math.sin(lat_rad) +
               math.cos(dec_rad) * math.cos(lat_rad) * math.cos(ha_rad))
    alt_rad = math.asin(sin_alt)

    # Calculate azimuth
    cos_az = ((math.sin(dec_rad) - math.sin(alt_rad) * math.sin(lat_rad)) /
              (math.cos(alt_rad) * math.cos(lat_rad)))
    cos_az = max(-1, min(1, cos_az))
    az_rad = math.acos(cos_az)

    # Adjust azimuth quadrant
    if math.sin(ha_rad) > 0:
        az_rad = 2 * math.pi - az_rad

    alt_deg = math.degrees(alt_rad)
    az_deg = math.degrees(az_rad)

    return alt_deg, az_deg

In [ ]:
def galactic_to_equatorial(l_deg, b_deg):
    """
    Convert Galactic coordinates to Equatorial (J2000).
    """
    # Galactic coordinate system constants (J2000)
    ra_ngp = math.radians(192.85948)
    dec_ngp = math.radians(27.12825)
    l_ncp = math.radians(122.93192)

    l_rad = math.radians(l_deg)
    b_rad = math.radians(b_deg)

    # Calculate declination
    sin_dec = (math.sin(dec_ngp) * math.sin(b_rad) +
               math.cos(dec_ngp) * math.cos(b_rad) * math.cos(l_ncp - l_rad))
    dec_rad = math.asin(sin_dec)

    # Calculate right ascension
    y = math.cos(b_rad) * math.sin(l_ncp - l_rad)
    x = (math.cos(dec_ngp) * math.sin(b_rad) -
         math.sin(dec_ngp) * math.cos(b_rad) * math.cos(l_ncp - l_rad))

    ra_rad = ra_ngp + math.atan2(y, x)

    ra_hours = (math.degrees(ra_rad) / 15) % 24
    dec_deg = math.degrees(dec_rad)

    return ra_hours, dec_deg

## Test 1: Sun Position

In [ ]:
# Test dates spanning different seasons
test_dates = [
    datetime(2024, 3, 20, 12, 0, 0, tzinfo=timezone.utc),  # Spring equinox
    datetime(2024, 6, 21, 12, 0, 0, tzinfo=timezone.utc),  # Summer solstice
    datetime(2024, 9, 22, 12, 0, 0, tzinfo=timezone.utc),  # Autumn equinox
    datetime(2024, 12, 21, 12, 0, 0, tzinfo=timezone.utc), # Winter solstice
    datetime(2026, 3, 13, 14, 30, 0, tzinfo=timezone.utc), # Current date
]

print("Sun Position Comparison")
print("=" * 80)
print(f"{'Date':<22} {'SRT RA':>10} {'Astropy RA':>12} {'RA Err':>10} {'SRT Dec':>10} {'Astropy Dec':>12} {'Dec Err':>10}")
print("-" * 80)

sun_ra_errors = []
sun_dec_errors = []

for dt in test_dates:
    # SRT calculation
    srt_ra, srt_dec = get_sun_position(dt)
    
    # Astropy calculation
    t = Time(dt)
    sun = get_sun(t)
    astropy_ra = sun.ra.hour
    astropy_dec = sun.dec.deg
    
    # Calculate errors (handle RA wrap-around)
    ra_err = srt_ra - astropy_ra
    if ra_err > 12:
        ra_err -= 24
    elif ra_err < -12:
        ra_err += 24
    ra_err_arcmin = ra_err * 15 * 60  # Convert hours to arcminutes
    
    dec_err = srt_dec - astropy_dec
    dec_err_arcmin = dec_err * 60  # Convert degrees to arcminutes
    
    sun_ra_errors.append(abs(ra_err_arcmin))
    sun_dec_errors.append(abs(dec_err_arcmin))
    
    print(f"{dt.strftime('%Y-%m-%d %H:%M'):<22} {srt_ra:>10.4f}h {astropy_ra:>10.4f}h {ra_err_arcmin:>+9.2f}' {srt_dec:>+10.4f}° {astropy_dec:>+10.4f}° {dec_err_arcmin:>+9.2f}'")

print("-" * 80)
print(f"Mean absolute error: RA = {np.mean(sun_ra_errors):.2f} arcmin, Dec = {np.mean(sun_dec_errors):.2f} arcmin")
print(f"Max absolute error:  RA = {np.max(sun_ra_errors):.2f} arcmin, Dec = {np.max(sun_dec_errors):.2f} arcmin")

## Test 2: Moon Position

In [ ]:
# Test dates for Moon (different lunar phases)
moon_test_dates = [
    datetime(2024, 1, 11, 12, 0, 0, tzinfo=timezone.utc),  # New moon
    datetime(2024, 1, 18, 12, 0, 0, tzinfo=timezone.utc),  # First quarter
    datetime(2024, 1, 25, 12, 0, 0, tzinfo=timezone.utc),  # Full moon
    datetime(2024, 2, 2, 12, 0, 0, tzinfo=timezone.utc),   # Last quarter
    datetime(2026, 3, 13, 14, 30, 0, tzinfo=timezone.utc), # Current date
    datetime(2024, 6, 15, 6, 0, 0, tzinfo=timezone.utc),   # Random date 1
    datetime(2024, 9, 20, 18, 0, 0, tzinfo=timezone.utc),  # Random date 2
]

print("Moon Position Comparison")
print("=" * 80)
print(f"{'Date':<22} {'SRT RA':>10} {'Astropy RA':>12} {'RA Err':>10} {'SRT Dec':>10} {'Astropy Dec':>12} {'Dec Err':>10}")
print("-" * 80)

moon_ra_errors = []
moon_dec_errors = []

for dt in moon_test_dates:
    # SRT calculation
    srt_ra, srt_dec = get_moon_position(dt)
    
    # Astropy calculation
    t = Time(dt)
    moon = get_body('moon', t)
    astropy_ra = moon.ra.hour
    astropy_dec = moon.dec.deg
    
    # Calculate errors
    ra_err = srt_ra - astropy_ra
    if ra_err > 12:
        ra_err -= 24
    elif ra_err < -12:
        ra_err += 24
    ra_err_arcmin = ra_err * 15 * 60
    
    dec_err = srt_dec - astropy_dec
    dec_err_arcmin = dec_err * 60
    
    moon_ra_errors.append(abs(ra_err_arcmin))
    moon_dec_errors.append(abs(dec_err_arcmin))
    
    print(f"{dt.strftime('%Y-%m-%d %H:%M'):<22} {srt_ra:>10.4f}h {astropy_ra:>10.4f}h {ra_err_arcmin:>+9.2f}' {srt_dec:>+10.4f}° {astropy_dec:>+10.4f}° {dec_err_arcmin:>+9.2f}'")

print("-" * 80)
print(f"Mean absolute error: RA = {np.mean(moon_ra_errors):.2f} arcmin, Dec = {np.mean(moon_dec_errors):.2f} arcmin")
print(f"Max absolute error:  RA = {np.max(moon_ra_errors):.2f} arcmin, Dec = {np.max(moon_dec_errors):.2f} arcmin")

## Test 3: RA/Dec to Alt/Az Conversion

In [ ]:
# Acre Road Observatory location
OBSERVER_LAT = 55.9
OBSERVER_LON = -4.3

location = EarthLocation(lat=OBSERVER_LAT*u.deg, lon=OBSERVER_LON*u.deg, height=50*u.m)

# Test objects at different positions
test_objects = [
    ("Polaris", 2.53, 89.26),           # Near NCP
    ("Vega", 18.62, 38.78),             # Bright star
    ("Betelgeuse", 5.92, 7.41),         # Near equator
    ("Sirius", 6.75, -16.72),           # Southern
    ("Galactic Center", 17.76, -29.0),  # Sgr A*
]

dt = datetime(2026, 3, 13, 20, 0, 0, tzinfo=timezone.utc)  # Evening observation
t = Time(dt)

# Disable atmospheric refraction for fair comparison
altaz_frame = AltAz(obstime=t, location=location, pressure=0)

print(f"Alt/Az Conversion Test at {dt.strftime('%Y-%m-%d %H:%M')} UTC")
print(f"Location: {OBSERVER_LAT}°N, {OBSERVER_LON}°E")
print("(Astropy with pressure=0, no atmospheric refraction)")
print("=" * 90)
print(f"{'Object':<18} {'SRT Alt':>10} {'Astropy Alt':>12} {'Alt Err':>10} {'SRT Az':>10} {'Astropy Az':>12} {'Az Err':>10}")
print("-" * 90)

alt_errors = []
az_errors = []

for name, ra, dec in test_objects:
    # SRT calculation
    srt_alt, srt_az = ra_dec_to_alt_az(ra, dec, OBSERVER_LAT, OBSERVER_LON, dt)
    
    # Astropy calculation (no refraction)
    coord = SkyCoord(ra=ra*u.hourangle, dec=dec*u.deg, frame='icrs')
    altaz = coord.transform_to(altaz_frame)
    astropy_alt = altaz.alt.deg
    astropy_az = altaz.az.deg
    
    alt_err = (srt_alt - astropy_alt) * 60  # arcmin
    az_err = srt_az - astropy_az
    if az_err > 180:
        az_err -= 360
    elif az_err < -180:
        az_err += 360
    az_err *= 60  # arcmin
    
    alt_errors.append(abs(alt_err))
    # Only include azimuth errors when altitude > 0 (above horizon)
    if astropy_alt > 0:
        az_errors.append(abs(az_err))
    
    # Mark objects below horizon with *
    below_horizon = "*" if astropy_alt <= 0 else " "
    print(f"{name:<18} {srt_alt:>+10.3f}° {astropy_alt:>+10.3f}° {alt_err:>+9.2f}' {srt_az:>10.3f}° {astropy_az:>10.3f}° {az_err:>+9.2f}'{below_horizon}")

print("-" * 90)
print(f"Mean absolute error: Alt = {np.mean(alt_errors):.2f} arcmin, Az = {np.mean(az_errors):.2f} arcmin")
print(f"Max absolute error:  Alt = {np.max(alt_errors):.2f} arcmin, Az = {np.max(az_errors):.2f} arcmin")
print(f"* Azimuth errors excluded for objects below horizon (Alt <= 0)")

## Debug: Check Sidereal Time and Coordinate Frame

Let's verify the SRT code and astropy are using the same sidereal time and that we understand the coordinate frames.

In [ ]:
# Debug comparison of SRT vs Astropy
dt = datetime(2026, 3, 13, 20, 0, 0, tzinfo=timezone.utc)
t = Time(dt)

print("=" * 60)
print("LOCATION CHECK")
print("=" * 60)
print(f"SRT:     lat = {OBSERVER_LAT}°, lon = {OBSERVER_LON}°")
print(f"Astropy: lat = {location.lat.deg}°, lon = {location.lon.deg}°")

print("\n" + "=" * 60)
print("SIDEREAL TIME CHECK")
print("=" * 60)

# SRT sidereal time
jd = julian_date(dt.year, dt.month, dt.day, dt.hour, dt.minute, dt.second)
srt_gmst = gmst(jd)
srt_lst = local_sidereal_time(jd, OBSERVER_LON)

# Astropy sidereal time
astropy_gmst = t.sidereal_time('mean', longitude=0).hour
astropy_lst = t.sidereal_time('mean', longitude=OBSERVER_LON*u.deg).hour

print(f"Julian Date: {jd}")
print(f"")
print(f"SRT GMST:              {srt_gmst:.6f} hours")
print(f"Astropy GMST (mean):   {astropy_gmst:.6f} hours")
print(f"Difference:            {(srt_gmst - astropy_gmst)*3600:.2f} seconds")

print("\n" + "=" * 60)
print("PRECESSION CHECK FOR VEGA (J2000: RA=18.62h, Dec=38.78°)")
print("=" * 60)
ra_vega = 18.62  # hours (J2000)
dec_vega = 38.78  # degrees (J2000)

# SRT precession
srt_ra_now, srt_dec_now = precess_j2000_to_date(ra_vega, dec_vega, jd)

# Astropy precession (ICRS -> TETE)
from astropy.coordinates import TETE
coord_icrs = SkyCoord(ra=ra_vega*u.hourangle, dec=dec_vega*u.deg, frame='icrs')
coord_tete = coord_icrs.transform_to(TETE(obstime=t))

print(f"J2000 RA/Dec:          {ra_vega:.4f}h, {dec_vega:.4f}°")
print(f"SRT precessed:         {srt_ra_now:.4f}h, {srt_dec_now:.4f}°")
print(f"Astropy TETE:          {coord_tete.ra.hour:.4f}h, {coord_tete.dec.deg:.4f}°")
print(f"RA difference:         {(srt_ra_now - coord_tete.ra.hour)*15*60:.2f} arcmin")
print(f"Dec difference:        {(srt_dec_now - coord_tete.dec.deg)*60:.2f} arcmin")

print("\n" + "=" * 60)
print("FULL TRANSFORM CHECK FOR VEGA (J2000 -> Alt/Az)")
print("=" * 60)

# SRT calculation (now with precession)
srt_alt, srt_az = ra_dec_to_alt_az(ra_vega, dec_vega, OBSERVER_LAT, OBSERVER_LON, dt)
print(f"SRT:                   Alt = {srt_alt:.4f}°, Az = {srt_az:.4f}°")

# Astropy with ICRS (J2000) - should now match!
altaz_frame = AltAz(obstime=t, location=location, pressure=0)
altaz_icrs = coord_icrs.transform_to(altaz_frame)
print(f"Astropy (ICRS->AltAz): Alt = {altaz_icrs.alt.deg:.4f}°, Az = {altaz_icrs.az.deg:.4f}°")

alt_err = (srt_alt - altaz_icrs.alt.deg) * 60
az_err = (srt_az - altaz_icrs.az.deg) * 60
print(f"\nDifference:            Alt = {alt_err:.2f} arcmin, Az = {az_err:.2f} arcmin")

## Azimuth Error vs Alt/Az Position

This plot shows how azimuth conversion error varies across the sky. We sample a grid of RA/Dec positions and plot the resulting azimuth error as a function of the computed Alt/Az.

In [ ]:
import matplotlib.pyplot as plt
from scipy.interpolate import griddata

# Sample a dense grid of RA/Dec positions
ra_values = np.linspace(0, 24, 100)  # Dense sampling in RA
dec_values = np.linspace(-30, 90, 100)  # Dense sampling in Dec

dt = datetime(2026, 3, 13, 20, 0, 0, tzinfo=timezone.utc)
t = Time(dt)

# Create AltAz frame WITHOUT atmospheric refraction for fair comparison
# (astropy includes refraction by default)
altaz_frame = AltAz(obstime=t, location=location, pressure=0)

# Collect data points
alt_list = []
az_list = []
az_err_list = []
pointing_err_list = []

for ra in ra_values:
    for dec in dec_values:
        # SRT calculation
        srt_alt, srt_az = ra_dec_to_alt_az(ra, dec, OBSERVER_LAT, OBSERVER_LON, dt)
        
        # Astropy calculation (no refraction)
        coord = SkyCoord(ra=ra*u.hourangle, dec=dec*u.deg, frame='icrs')
        altaz = coord.transform_to(altaz_frame)
        astropy_alt = altaz.alt.deg
        astropy_az = altaz.az.deg
        
        # Only include points above horizon
        if astropy_alt > 0:
            # Azimuth error
            az_err = srt_az - astropy_az
            if az_err > 180:
                az_err -= 360
            elif az_err < -180:
                az_err += 360
            az_err *= 60  # Convert to arcmin
            
            # Calculate true angular separation (pointing error)
            # Using spherical law of cosines
            alt1_rad = math.radians(srt_alt)
            alt2_rad = math.radians(astropy_alt)
            az1_rad = math.radians(srt_az)
            az2_rad = math.radians(astropy_az)
            
            cos_sep = (math.sin(alt1_rad) * math.sin(alt2_rad) + 
                       math.cos(alt1_rad) * math.cos(alt2_rad) * math.cos(az1_rad - az2_rad))
            cos_sep = max(-1, min(1, cos_sep))  # Clamp for numerical stability
            separation = math.degrees(math.acos(cos_sep)) * 60  # arcmin
            
            alt_list.append(astropy_alt)
            az_list.append(astropy_az)
            az_err_list.append(abs(az_err))
            pointing_err_list.append(separation)

# Create regular grid with 0.5 degree increments
alt_grid = np.arange(0.5, 90.5, 0.5)
az_grid = np.arange(0.5, 360.5, 0.5)
AZ_GRID, ALT_GRID = np.meshgrid(az_grid, alt_grid)

# Interpolate errors onto regular grid
points = np.column_stack((az_list, alt_list))
az_err_grid = griddata(points, az_err_list, (AZ_GRID, ALT_GRID), method='linear')
pointing_err_grid = griddata(points, pointing_err_list, (AZ_GRID, ALT_GRID), method='linear')

# Create the 2D plots
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Azimuth error
ax1 = axes[0]
pcm1 = ax1.pcolormesh(az_grid, alt_grid, az_err_grid, cmap='hot_r', shading='auto')
cbar1 = plt.colorbar(pcm1, ax=ax1, label='Azimuth Error (arcmin)')
ax1.set_xlabel('Azimuth (degrees)')
ax1.set_ylabel('Altitude (degrees)')
ax1.set_title('Azimuth Error (coordinate difference)')
ax1.set_xlim(0, 360)
ax1.set_ylim(0, 90)
contours1 = ax1.contour(AZ_GRID, ALT_GRID, az_err_grid, levels=[20, 40, 60, 80, 100], colors='black', linewidths=0.5)
ax1.clabel(contours1, inline=True, fontsize=8, fmt='%.0f\'')

# Plot 2: True pointing error
ax2 = axes[1]
pcm2 = ax2.pcolormesh(az_grid, alt_grid, pointing_err_grid, cmap='hot_r', shading='auto')
cbar2 = plt.colorbar(pcm2, ax=ax2, label='Pointing Error (arcmin)')
ax2.set_xlabel('Azimuth (degrees)')
ax2.set_ylabel('Altitude (degrees)')
ax2.set_title('True Pointing Error (angular separation)')
ax2.set_xlim(0, 360)
ax2.set_ylim(0, 90)
contours2 = ax2.contour(AZ_GRID, ALT_GRID, pointing_err_grid, levels=[5, 10, 15, 20, 25], colors='black', linewidths=0.5)
ax2.clabel(contours2, inline=True, fontsize=8, fmt='%.0f\'')

plt.suptitle(f'{dt.strftime("%Y-%m-%d %H:%M")} UTC, Location: {OBSERVER_LAT}°N, {OBSERVER_LON}°E (no refraction)', y=1.02)
plt.tight_layout()
plt.show()

print(f"\nStatistics for Alt > 0 (astropy with pressure=0, no refraction):")
print(f"  Points sampled: {len(az_err_list)}")
print(f"\n  Azimuth Error:")
print(f"    Mean: {np.mean(az_err_list):.2f} arcmin")
print(f"    Max:  {np.max(az_err_list):.2f} arcmin")
print(f"\n  True Pointing Error:")
print(f"    Mean: {np.mean(pointing_err_list):.2f} arcmin")
print(f"    Max:  {np.max(pointing_err_list):.2f} arcmin")

## Test 4: Galactic to Equatorial Conversion

In [ ]:
# Test galactic coordinates
galactic_tests = [
    ("Galactic Center", 0.0, 0.0),
    ("Galactic Anticenter", 180.0, 0.0),
    ("North Galactic Pole", 0.0, 90.0),
    ("South Galactic Pole", 0.0, -90.0),
    ("Cygnus X", 80.0, 0.0),
    ("Cas A region", 111.7, -2.1),
]

print("Galactic to Equatorial Conversion Test")
print("=" * 90)
print(f"{'Object':<20} {'l':>8} {'b':>8} {'SRT RA':>10} {'Astropy RA':>12} {'SRT Dec':>10} {'Astropy Dec':>12}")
print("-" * 90)

gal_ra_errors = []
gal_dec_errors = []

for name, l, b in galactic_tests:
    # SRT calculation
    srt_ra, srt_dec = galactic_to_equatorial(l, b)
    
    # Astropy calculation
    coord = SkyCoord(l=l*u.deg, b=b*u.deg, frame='galactic')
    icrs = coord.icrs
    astropy_ra = icrs.ra.hour
    astropy_dec = icrs.dec.deg
    
    ra_err = srt_ra - astropy_ra
    if ra_err > 12:
        ra_err -= 24
    elif ra_err < -12:
        ra_err += 24
    ra_err_arcmin = ra_err * 15 * 60
    
    dec_err = (srt_dec - astropy_dec) * 60
    
    gal_ra_errors.append(abs(ra_err_arcmin))
    gal_dec_errors.append(abs(dec_err))
    
    print(f"{name:<20} {l:>8.1f}° {b:>+8.1f}° {srt_ra:>10.4f}h {astropy_ra:>10.4f}h {srt_dec:>+10.4f}° {astropy_dec:>+10.4f}°")

print("-" * 90)
print(f"Mean absolute error: RA = {np.mean(gal_ra_errors):.4f} arcmin, Dec = {np.mean(gal_dec_errors):.4f} arcmin")

## Test 5: Julian Date Calculation

In [ ]:
# Test Julian Date calculation against known values
jd_tests = [
    ((2000, 1, 1, 12, 0, 0), 2451545.0, "J2000.0 epoch"),
    ((1858, 11, 17, 0, 0, 0), 2400000.5, "MJD epoch"),
    ((2024, 3, 20, 3, 6, 0), 2460389.62917, "Vernal equinox 2024"),
]

print("Julian Date Calculation Test")
print("=" * 70)
print(f"{'Description':<25} {'Calculated JD':>18} {'Expected JD':>18} {'Error':>10}")
print("-" * 70)

for (y, m, d, h, mi, s), expected, desc in jd_tests:
    calculated = julian_date(y, m, d, h, mi, s)
    error = calculated - expected
    print(f"{desc:<25} {calculated:>18.5f} {expected:>18.5f} {error:>+10.6f}")

## Summary

In [ ]:
print("\n" + "=" * 60)
print("ACCURACY SUMMARY")
print("=" * 60)
print(f"\nSun Position:")
print(f"  Mean error: {np.mean(sun_ra_errors):.2f}' RA, {np.mean(sun_dec_errors):.2f}' Dec")
print(f"  Max error:  {np.max(sun_ra_errors):.2f}' RA, {np.max(sun_dec_errors):.2f}' Dec")

print(f"\nMoon Position:")
print(f"  Mean error: {np.mean(moon_ra_errors):.2f}' RA, {np.mean(moon_dec_errors):.2f}' Dec")
print(f"  Max error:  {np.max(moon_ra_errors):.2f}' RA, {np.max(moon_dec_errors):.2f}' Dec")

print(f"\nAlt/Az Conversion:")
print(f"  Mean error: {np.mean(alt_errors):.2f}' Alt, {np.mean(az_errors):.2f}' Az")
print(f"  Max error:  {np.max(alt_errors):.2f}' Alt, {np.max(az_errors):.2f}' Az")
print(f"  (Az errors only for objects with Alt > 0)")

print(f"\nGalactic Conversion:")
print(f"  Mean error: {np.mean(gal_ra_errors):.4f}' RA, {np.mean(gal_dec_errors):.4f}' Dec")

# Typical SRT beam width at 1420 MHz
beam_width = 7 * 60  # arcmin (assuming ~7 degree beam)
print(f"\n" + "-" * 60)
print(f"Typical SRT beam width at 1420 MHz: ~{beam_width/60:.0f}° ({beam_width:.0f} arcmin)")
print(f"All errors are well within the beam width.")
print("=" * 60)